# Nucleoplasm segmentation — napari step visualisation
Load one example, view every intermediate step as a napari layer, tune the parameters at the top, re-run.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import napari
from aicsimageio import AICSImage
from skimage.morphology import binary_erosion, disk

## Parameters to optimise

In [ ]:
# --- paths ---
input_file = Path("EXAMPLE.ome.tiff")  # bare filename
input_dir = Path("/srv/scratch/berrylab/z3536241/NikonSpinningDisk/260515_mACPOLR2A_CRC_EU")
label_image_dir = Path("/path/to/labels")
probmap_dir = Path("/srv/scratch/berrylab/z3532965/tmp")

# --- tunables ---
prob_threshold = 128       # uint8 probability map (0-255)
erosion_radius_1 = 1       # first erosion (nucleus)
erosion_radius_2 = 1       # second erosion (clear nucleolus margin)

## Helper

In [ ]:
def erode_labels(label_img, selem):
    """Erode each label independently so touching labels separate at their shared boundary."""
    out = np.zeros_like(label_img)
    for lbl in np.unique(label_img):
        if lbl == 0:
            continue
        out[binary_erosion(label_img == lbl, selem)] = lbl
    return out

## Load images

In [ ]:
intensity_image = AICSImage(input_dir / input_file)

labels = AICSImage(str(label_image_dir) + "/labels_" + input_file.name)

nucleoplasm_prob = AICSImage(
    str(probmap_dir) + "/" + input_file.stem + "_Probabilities.tiff"
)

print("intensity:", intensity_image.dims)
print("labels:", labels.dims)
print("prob:", nucleoplasm_prob.dims, nucleoplasm_prob.dtype)

assert nucleoplasm_prob.dtype == np.uint8, f"expected uint8, got {nucleoplasm_prob.dtype}"

## Run each step (re-run this + the napari cell after changing params)

In [ ]:
# raw inputs
nucleus_labels = labels.get_image_data("YX", C=0).astype(np.int32)
prob = nucleoplasm_prob.get_image_data("YX")

# step 1: threshold -> nucleoplasm mask
nucleoplasm_mask = prob > prob_threshold

# step 2: erode nucleus (per-label)
eroded_nuclei = erode_labels(nucleus_labels, disk(erosion_radius_1))

# step 3: mask eroded nucleus by nucleoplasm
nucleoplasm_labels = np.where(nucleoplasm_mask, eroded_nuclei, 0).astype(np.int32)

# step 4: erode again to clear nucleolus margin
nucleoplasm_final = erode_labels(nucleoplasm_labels, disk(erosion_radius_2)).astype(np.int32)

print("nuclei:", len(np.unique(nucleus_labels)) - 1,
      "| final nucleoplasm regions:", len(np.unique(nucleoplasm_final)) - 1)

## Visualise in napari

In [ ]:
# pick an intensity channel for reference background
ref = intensity_image.get_image_data("YX", C=0)

viewer = napari.Viewer()
viewer.add_image(ref, name="intensity (C0)", colormap="gray", blending="additive")
viewer.add_image(prob, name="prob map", colormap="magma", blending="additive", visible=False)
viewer.add_labels(nucleus_labels, name="1. nucleus labels")
viewer.add_labels(eroded_nuclei, name="2. eroded nuclei")
viewer.add_labels(nucleoplasm_mask.astype(np.uint8), name="3. nucleoplasm mask")
viewer.add_labels(nucleoplasm_labels, name="4. masked (pre 2nd erosion)")
viewer.add_labels(nucleoplasm_final, name="5. final nucleoplasm")

# napari.run()  # uncomment if running as a plain script rather than in Jupyter

### Tuning loop
Adjust `prob_threshold`, `erosion_radius_1`, `erosion_radius_2` above, re-run the **Run each step** cell, then toggle layers in the open napari window (no need to re-add). Watch layer 5 against the prob map / nucleolus to confirm no overlap and that touching nuclei are separated.